In [ ]:
!pip freeze > requirements.txt

In [ ]:
yaml_config =  """
               model_path: "sd-legacy/stable-diffusion-v1-5"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
import torch
from omegaconf import OmegaConf
from diffusers import StableDiffusionPipeline

class ImageGenerater:
  def __init__(self, config_path):
    self.config = OmegaConf.load(config_path)
    self.pipe = StableDiffusionPipeline.from_pretrained(
        self.config.model_path,
        torch_dtype=torch.float16
    )
    self.pipe = self.pipe.to("cuda")

  def __call__(self, prompt):
    return self.pipe(prompt).images[0]

In [ ]:
# load model 
generater = ImageGenerater("./config.yaml")

In [ ]:
# test in local
generater("Create a robot.")

In [ ]:
# server
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import psutil
import platform
import torch
import io
import base64
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# help
@app.get('/')
async def intro():
  return {
      "description": "Create image from text",
      "endpoints": {
          "GET /health": "Check the system's operational status.",
          "POST /generate": "Receive prompt to create image and return image."
      }
  }
#check health
@app.get('/health')
async def health():
  health_status = {}
  # 1. GPU Health
  if torch.cuda.is_available():
    health_status["gpu"] = {
        "status": "available",
        "name": torch.cuda.get_device_name(0),
        "memory_allocated_mb": round(torch.cuda.memory_allocated(0) / 1024**2, 2),
        "memory_reserved_mb": round(torch.cuda.memory_reserved(0) / 1024**2, 2)
    }
  else:
        health_status["gpu"] = {"status": "not_available", "warning": "Running on CPU"}

  # 2. CPU & RAM Health
  ram = psutil.virtual_memory()
  health_status["system"] = {
      "cpu_name": platform.processor(),
      "cpu_usage_percent": psutil.cpu_percent(),
      "ram_usage_percent": ram.percent,
      "ram_available_gb": round(ram.available / 1024**3, 2),
      "ram_total_gb": round(ram.total / 1024**3, 2)
  }

  # 3. Disk Health
  disk = psutil.disk_usage('/')
  health_status["disk"] = {
      "total_gb": round(disk.total / 1024**3, 2),
      "used_gb": round(disk.used / 1024**3, 2),
      "free_gb": round(disk.free / 1024**3, 2),
      "percent": disk.percent
  }
  return health_status
#generate image
@app.post('/generate')
async def generateImage(prompt: str):
  if not prompt:
      return JSONResponse(content={
          "status": "fail",
          "status_code": 400,
          "message": "prompt is empty"
      })
  # generate image
  image = generater(prompt)
  #Save the image to the buffer as a PNG.
  img_byte_arr = io.BytesIO()
  image.save(img_byte_arr, format='PNG')
  img_data = img_byte_arr.getvalue() # Lấy toàn bộ dữ liệu binary

  # 3. Mã hóa binary sang chuỗi Base64
  base64_encoded_result = base64.b64encode(img_data).decode('utf-8')
  return JSONResponse(content={
      "status": "success",
      "status_code": 200,
      "prompt": prompt,
      "image_base64": f"data:image/png;base64,{base64_encoded_result}"
  })

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(5)

In [ ]:
# Create a public link by running a command in the terminal.
# "ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io"